<a href="https://colab.research.google.com/github/IvanIri/Procesamiento-del-Habla/blob/main/TP_Semana1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Iván Irigoyen**

In [26]:
!pip install pymupdf pdfplumber nltk matplotlib pandas -q

In [27]:
import requests
import fitz  # PyMuPDF
import pdfplumber
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import re
import string
from collections import Counter

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [28]:
# Descarga de PDF

pdf_url = "https://www.imf.org/-/media/files/publications/weo/2023/october/spanish/text.pdf"
pdf_path = "text.pdf"

response = requests.get(pdf_url)
with open(pdf_path, "wb") as f:
  f.write(response.content)

print("PDF descargado exitosamente.")

PDF descargado exitosamente.


In [29]:
# Abrimos el documento

doc = fitz.open(pdf_path)

# Obtenemos los metadatos
metadata = doc.write
num_pag = doc.page_count

doc.metadata

{'format': 'PDF 1.6',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'creator': 'Adobe InDesign 19.0 (Macintosh)',
 'producer': 'Adobe PDF Library 17.0',
 'creationDate': "D:20231205233201+01'00'",
 'modDate': "D:20231205233310+01'00'",
 'trapped': '',
 'encryption': None}

Vemos mucha información faltantes con este analisis preliminar de los metadatos.

Solo obtenemos el formato "PDF 1.6", que fue hecho en "Adobe 19.0" y sus fechas de creación y actualización.

In [30]:
# Extraemos el texto crudo de las primeras 5 paginas.

texto_pag = []

for i in range(min(5, doc.page_count)):
  pag = doc.load_page(i)
  texto = pag.get_text("text")

  texto_pag.append(texto)

  print(f"\n{'='*40}")
  print(f"PAGINA {i+1}")
  print(f"{'='}*\n")
  print(texto[0:3000]) # le doy un maximo de caracteres a la salida.


PAGINA 1
=*

PERSPECTIVAS
DE LA ECONOMÍA
MUNDIAL
OCT
2023
Abordar las divergencias mundiales
FONDO MONETARIO INTERNACIONAL


PAGINA 2
=*

PERSPECTIVAS
DE LA ECONOMÍA
MUNDIAL
Abordar las divergencias mundiales
OCT
2023
FONDO MONETARIO INTERNACIONAL


PAGINA 3
=*

©2023 International Monetary Fund 
Edición en español ©2023 Fondo Monetario Internacional
 
Edición en inglés 
Portada y diseño: División de Soluciones Creativas, CSF, FMI
Composición: Absolute Service, Inc.; y AGS, una firma de RR Donnelley Company
Edición en español 
Sección de Español y Portugués 
Servicios Lingüísticos del FMI 
Departamento de Servicios Corporativos e Instalaciones del FMI
 
Cataloging-in-Publication Data
IMF Library
Names: International Monetary Fund.
Title: World economic outlook (International Monetary Fund)
Other titles: WEO | Occasional paper (International Monetary Fund) | World economic and 
financial surveys.
Description: Washington, DC : International Monetary Fund, 1980- | Semiannual | Some 
issu

Obtuvimos las primeras 5 paginas, algúnas contienen mas texto que otras. En el caso de las primeras esto sucede por q son titulos o las paginas contienen imagenes. Luego viene un gran texto que es introducción o presentación. Y el resto de paginas son un indice de los temas que contiene el pdf.

In [31]:
# Analizamos los separadores

muestra = texto_pag[4]
print(repr(muestra))

'PERSPECTIVAS DE LA ECONOMÍA MUNDIAL: ABORDAR LAS DIVERGENCIAS MUNDIALES\niv\t\nFondo Monetario Internacional | Octubre de 2023\nResumen e implicaciones para las políticas\t\n100\nRecuadro 3.1. Tensiones en el comercio internacional de materias primas:  \nObservaciones derivadas de los datos sobre buques cisterna\t\n103\nRecuadro 3.2. La fragmentación de los mercados de materias primas  \ncon una óptica histórica: Muchos matices de gris\t\n104\nRecuadro 3.3. Variedad de los efectos económicos de la fragmentación  \nde los mercados de materias primas\t\n105\nReferencias\t\n107\nApéndice estadístico\t\n111\nSupuestos\t\n111\nNovedades\t\n111\nDatos y convenciones\t\n112\nNotas sobre los países\t\n113\nClasificación de los países\t\n115\nCaracterísticas generales y composición de los grupos que conforman  \nla clasificación del informe WEO\t\n116\nCuadro A. Clasificación según los grupos utilizados en Perspectivas  \nde la economía mundial y la participación de cada grupo en el PIB agrega

Acá vemos que se utilizan los separadores "\n" y "\t" para los saltos de lineas.

Estos los vemos mucho por que justo estoy analizando las paginas que contienen los indices del documento. por lo que luego de cada titulo nos encontramos con un separador que hacen que el texto del indice este en forma de "lista".

No observo el uso de otros separadores.

In [32]:
# Extracción de datos tabulares

# pagina 33 contiene datos tabulares

with pdfplumber.open(pdf_path) as pdf:
  pag = pdf.pages[33]
  tablas = pag.extract_tables()

tabla = tablas[0]
df_tabla = pd.DataFrame(tabla)

df_tabla.head()

,0,1,2,3,4,5,6,7
0,Proyecciones,None,,Diferencia con la\nActualización del informe\n...,None,,Diferencia con\nel informe WEO\nde abril de 20231,None
1,2023,2024,None,2023,2024,None,2023,2024
2,"2,9","3,2",,"0,0","0,3",,"0,0","0,1"
3,"1,5","1,5",None,"0,1","0,1",None,"0,4","–0,1"
4,"1,9","1,4",None,"0,5","0,3",None,"0,9","0,1"


Me trajo la tabla, pero no completa, no logro identificar varios titulos de las columnas y no trajo ningúno de los de las filas.

Esto puede ser por que no es una tabla tipica la del documento, no tiene los limites de cada fila y columna bien definidos

In [33]:
# Intento 2 de extración de datos tabulares

with pdfplumber.open(pdf_path) as pdf:
  pag = pdf.pages[33]

  tablas = pag.extract_table({
      "vertical_strategy": "lines",
      "horizontal_strategy": "lines"
  })

df_tabla = pd.DataFrame(tablas)
df_tabla.head()

,0,1,2,3,4,5,6,7
0,Proyecciones,None,,Diferencia con la\nActualización del informe\n...,None,,Diferencia con\nel informe WEO\nde abril de 20231,None
1,2023,2024,None,2023,2024,None,2023,2024
2,"2,9","3,2",,"0,0","0,3",,"0,0","0,1"
3,"1,5","1,5",None,"0,1","0,1",None,"0,4","–0,1"
4,"1,9","1,4",None,"0,5","0,3",None,"0,9","0,1"


Probamos otra forma dandole más indicaciones de como esta formada la tabla, pero aún así no nos trae la información de la tabla completamente.

Al no tener un formato convencional la tabla del documento las extracciones de datos tabulares se dificulta. No alcanza con "pdfplumber" para su extracción

In [34]:
# Inteto 3 de extracción de datos tabulares.
# Ahora probamos Tabula

!pip install tabula-py

In [35]:
import tabula

df_tablas = tabula.read_pdf(pdf_path, pages=33, multiple_tables=True)

for df in df_tablas:
    print(df.head(10))

            Unnamed: 0 Unnamed: 1    Unnamed: 2          Diferencia con la  \
0                  NaN        NaN           NaN  Actualización del informe   
1                  NaN        NaN  Proyecciones      WEO de julio de 20231   
2                  NaN       2022     2023 2024                  2023 2024   
3     Producto mundial        3,5       3,0 2,9                   0,0 –0,1   
4  Economías avanzadas        2,6       1,5 1,4                    0,0 0,0   
5       Estados Unidos        2,1       2,1 1,5                    0,3 0,5   
6        Zona del euro        3,3       0,7 1,2                  –0,2 –0,3   
7             Alemania        1,8      –0,5 0,9                  –0,2 –0,4   
8              Francia        2,5       1,0 1,3                    0,2 0,0   
9              Italia2        3,7       0,7 0,7                  –0,4 –0,2   

      Diferencia con  
0     el informe WEO  
1  de abril de 20231  
2          2023 2024  
3           0,2 –0,1  
4            0,2 0,0  
5  

Observamos que Tabula funciono muchisimo mejor. Trajo más información de las celdas y completo tanto los titulos de las columnas como los de las filas.

los valores "NaN" que se observan corresponden a espacios en blanco de la tabla.

Los "Unnamed" hacen referencia a celdas sin información también, solo que las etiqueto diferente. Supongo que interpreto que solo la 1ra fila era el titulo de las columnas, cuando lo son las primeras 3/4 filas.

In [36]:
# Limpieza de las tablas

df_tabla.columns = [
    str(col).replace('Unnamed', '-')
    for col in df_tabla.columns
]

df_tabla.head(10)

,0,1,2,3,4,5,6,7
0,Proyecciones,None,,Diferencia con la\nActualización del informe\n...,None,,Diferencia con\nel informe WEO\nde abril de 20231,None
1,2023,2024,None,2023,2024,None,2023,2024
2,"2,9","3,2",,"0,0","0,3",,"0,0","0,1"
3,"1,5","1,5",None,"0,1","0,1",None,"0,4","–0,1"
4,"1,9","1,4",None,"0,5","0,3",None,"0,9","0,1"
5,"0,7","1,4",None,"–0,5","–0,1",None,"0,0","–0,4"
6,"–0,2","1,7",None,"–0,7","0,2",None,"–0,4","–0,1"
7,"1,0","1,5",None,"0,1","–0,1",None,"0,2","0,1"
8,"0,3","1,2",None,"–0,6","0,1",None,"–0,1","0,1"
9,"1,6","2,0",None,"–0,2","–0,2",None,"0,3","–0,1"


En esta parte de limpieza buscaba eliminar los "Unnamed" y los "Nan" que aparecen. Estos aparecen en celdas vacias y en celdas que interpreta como titulo. probe de varias formas que fui eliminando, pero no logre que me mantenga toda la info, en todos los casos me elimina la 1ra columna que contiene en cada fila a que hace referencia los valores de esa fila.

**NPL Basico**

In [37]:
!pip install nltk -q

In [38]:
import nltk
import string
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [39]:
# Chequeamos si la pagina tiene solo texto

pagina_narrativa = doc.load_page(83)
texto_narrativo = pagina_narrativa.get_text("text")

print(texto_narrativo[:2000])

65
CAPÍTULO 2 
Gestionar las expectativas: Inflación y política monetaria
Fondo Monetario Internacional | Octubre de 2023
Las estimaciones de los coeficientes representan la 
variación en la inflación asociada a un incremento 
de una desviación estándar del indicador de las 
expectativas indicado12. El primer hallazgo es que 
las expectativas de inflación a largo plazo tienen un 
poder de predicción menor que el de indicadores a 
corto plazo. Las expectativas inflacionarias a cinco 
años, tanto basadas en el mercado financiero como de 
los profesionales que elaboran pronósticos, presentan 
coeficientes estandarizados menores a los de otros 
indicadores (gráfico 2.5, dos conjuntos inferiores de 
cajas y flecos). Estos resultados se condicen con los 
de estudios recientes que hallan que las expectativas 
a largo plazo inciden menos en la inflación corriente 
(Werning, 2022; Hajdini, 2023). En segundo lugar, se 
observa una notable coherencia entre las expectativas 
de inflación a corto p

In [40]:
nltk.download('punkt_tab')

# Tokenización. Convertir el texto en minuscula

texto_narrativo = texto_narrativo.lower()
tokens = word_tokenize(texto_narrativo, language='spanish')

print(tokens[:50])

['65', 'capítulo', '2', 'gestionar', 'las', 'expectativas', ':', 'inflación', 'y', 'política', 'monetaria', 'fondo', 'monetario', 'internacional', '|', 'octubre', 'de', '2023', 'las', 'estimaciones', 'de', 'los', 'coeficientes', 'representan', 'la', 'variación', 'en', 'la', 'inflación', 'asociada', 'a', 'un', 'incremento', 'de', 'una', 'desviación', 'estándar', 'del', 'indicador', 'de', 'las', 'expectativas', 'indicado12', '.', 'el', 'primer', 'hallazgo', 'es', 'que', 'las']


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [41]:
# Limpieza. Eliminar las stopwords y signos de puntuación

stop_words = set(stopwords.words('spanish'))

tokens_limpios = [
    token for token in tokens
    if token not in stop_words
    and token not in string.punctuation
    and token.isalpha()
]

print(tokens_limpios[:50])

['capítulo', 'gestionar', 'expectativas', 'inflación', 'política', 'monetaria', 'fondo', 'monetario', 'internacional', 'octubre', 'estimaciones', 'coeficientes', 'representan', 'variación', 'inflación', 'asociada', 'incremento', 'desviación', 'estándar', 'indicador', 'expectativas', 'primer', 'hallazgo', 'expectativas', 'inflación', 'largo', 'plazo', 'poder', 'predicción', 'menor', 'indicadores', 'corto', 'plazo', 'expectativas', 'inflacionarias', 'cinco', 'años', 'basadas', 'mercado', 'financiero', 'profesionales', 'elaboran', 'pronósticos', 'presentan', 'coeficientes', 'estandarizados', 'menores', 'indicadores', 'gráfico', 'dos']


In [42]:
# Frecuencia de palabras

frecuencias = Counter(tokens_limpios)
top_15 = frecuencias.most_common(15)

print(top_15)

[('inflación', 24), ('expectativas', 23), ('economías', 14), ('plazo', 12), ('corto', 9), ('coeficientes', 8), ('mercados', 8), ('emergentes', 7), ('corriente', 6), ('menor', 5), ('rezagada', 5), ('octubre', 4), ('incremento', 4), ('largo', 4), ('inflacionarias', 4)]


In [43]:
# convertir en DataFrame para graficar

df_freq = pd.DataFrame(top_15, columns=['Palabra', 'Frecuencia'])

df_freq

,Palabra,Frecuencia
0,inflación,24
1,expectativas,23
2,economías,14
3,plazo,12
4,corto,9
5,coeficientes,8
6,mercados,8
7,emergentes,7
8,corriente,6
9,menor,5


In [44]:
# Grafico de barras de las palabras más frecuentes

import plotly.express as px

fig = px.bar(
    df_freq,
    x='Palabra',
    y='Frecuencia',
    title='Top 15 palabras más frecuentes',
    labels={'Palabra': 'Palabras', 'Frecuencia': 'Frecuencia'},
    text='Frecuencia'
)

fig.update_layout(
    xaxis_tickangle=45
)

fig.show()

No me sugirio ninguna palabra considerada stopwords. Por lo que puedo asegurar que la limpieza de este tipo de palabras funciono muy bien.